# 🏗️ Structural Engineering AI - Training Gemma 2 9B (L4 GPU)

Notebook ini disiapkan khusus untuk Milestone **18 April**. 

**Fitur:**
1. Hybrid Dataset (SNI Theory + Structural Analysis CoT).
2. Optimized for L4 GPU (Google Colab).
3. **Real-time Sync to Google Drive** (Checkpoints & Final Model).

## 1. Persiapan Environment

In [ ]:
# 1. Mount Google Drive (WAJIB untuk sinkronisasi hasil)
from google.colab import drive
drive.mount('/content/drive')

# 2. Clone Repository (Gunakan URL Repo Anda)
!git clone https://github.com/tech4iddev/COLAB_GEMMA_4.git /content/structural_ai
%cd /content/structural_ai/model_18_april

# 3. Install Unsloth & Dependencies
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install unsloth_zoo
!pip install --no-deps xformers trl peft accelerate bitsandbytes

## 2. Sinkronisasi Data (Master Pipeline)
Gunakan cell ini untuk menggabungkan seluruh file Markdown SNI dan soal hitungan sintetis menjadi satu file training: `dataset_final_18_april.jsonl`.

In [ ]:
!python master_pipeline.py

## 3. Proses Training (Langsung Sinkron ke Drive)
Script ini akan menyimpan checkpoint setiap 30 step langsung ke `/content/drive/MyDrive/Structural_AI_Project/outputs`.

In [ ]:
!python train_colab_L4.py

## 4. Testing & Inference (Debug)
Mari kita coba model yang sudah tersimpan di Drive.

In [ ]:
from unsloth import FastLanguageModel
import torch

# Load model hasil training dari Google Drive
model_path = "/content/drive/MyDrive/Structural_AI_Project/gemma2-9b-structural-18april"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_path,
    max_seq_length = 2048,
    load_in_4bit = True,
)
FastLanguageModel.for_inference(model)

def ask_ai(instruction, input_text=""):
    prompt_style = """<start_of_turn>user
{} 
{}<end_of_turn>
<start_of_turn>model
"""
    inputs = tokenizer([prompt_style.format(instruction, input_text)], return_tensors = "pt").to("cuda")
    outputs = model.generate(**inputs, max_new_tokens = 512, use_cache = True)
    return tokenizer.batch_decode(outputs)[0]

# TEST HITUNGAN
response = ask_ai("Hitung kapasitas tarik profil baja jika fy=250 MPa dan A=3000 mm2.")
print(response)